# 10 · Событие = ПЕРВАЯ неисправность серии + слой «через сколько дней»

**Цель** — два слоя прогноза на 6-часовом панэле:

1. **Детекция события по первому эпизоду серии.** Серия = неисправные бакеты
   канала с разрывом < 24ч. Считается, что «новое событие» — только **первая**
   неисправность серии; последующие неисправности — продолжение, для таргета
   неактуальны (`серия_старт`, `цель_серии_6ч/12ч/24ч/48ч`).
2. **Слой «через сколько дней»** — ор-динальный мультикласс по бин-классам
   расстояния до следующего события: ≤0.5д, 0.5–1д, 1–2д, 2–3д, 3–7д, 7–14д,
   14–30д, >30д/нет события (цензур). Ожидаемые дни = Σ p(класс)·mid(класс).

Разметка: `features.add_series_targets` (gap = 4 бакета = 24 ч).

In [ ]:
import sys
import pathlib

_HERE = pathlib.Path.cwd()
if _HERE.name == "notebooks":
    RESEARCH = _HERE.parent
else:
    RESEARCH = _HERE
if str(RESEARCH) not in sys.path:
    sys.path.insert(0, str(RESEARCH))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import data_utils as du
import features as fe

RECOMPUTE = False
print("dataset:", du.find_dataset_dir())

# 10.1 · Серийная разметка на 6-часовом панэле

`features.add_series_targets` помечает первый неисправный бакет серии как
`серия_старт=1` (остальные неисправности той же серии — продолжение) и строит:
`дни_до_серии` (до следующего старта, сутки; NaN — цензур) и бинарные цели
`цель_серии_6ч/12ч/24ч/48ч` (новое событие в ближайшие 1/2/4/8 бакетов).

In [ ]:
panel = fe.make_subdaily_panel(recompute=RECOMPUTE)
panel = fe.add_series_targets(panel)

# Ор-динальные классы «дней до события» (для второго слоя)
BINS_DAYS = [0.5, 1, 2, 3, 7, 14, 30]
MID = np.array([0.25, 0.75, 1.5, 2.5, 5.0, 10.5, 22.0, 45.0])
panel["дни_класс"] = np.clip(np.digitize(panel["дни_до_серии"], BINS_DAYS), 0, len(BINS_DAYS))
panel["дни_класс"] = panel["дни_класс"].fillna(len(BINS_DAYS)).astype(int)  # цензур -> «>30д»

print("Суб-суточный панэль:", panel.shape)
print(f"неисправных бакетов: {(panel['неисправностей'] > 0).sum():,} | "
      f"стартов серий (событий): {int(panel['серия_старт'].sum()):,} "
      f"({100 * panel['серия_старт'].mean():.2f}% строк)")
print("\nБаза новых событий (цель_серии_*, чистые дни):")
for n in fe.HORIZONS:
    y = panel[f"цель_серии_{n}"].dropna()
    print(f"  цель_серии_{n}: {100 * y.mean():6.3f}%")
d = panel["дни_до_серии"].dropna()
print(f"\nдней до след. события: медиана {np.median(d):.1f}, "
      f"q25 {np.quantile(d, .25):.1f}, q75 {np.quantile(d, .75):.1f}; "
      f"цензурировано {100 * panel['дни_до_серии'].isna().mean():.2f}%")
print("\nраспределение дней_класс:")
print(panel["дни_класс"].value_counts().sort_index().rename_axis("класс").to_string())

# 10.2 · Временной сплит и признаки

Те же признаки, что в 09 (+ категории). Цели — серийные (`цель_серии_*`),
классы дней (`дни_класс`) вынесены из X.

In [ ]:
panel["ид_объект_code"] = panel["ид_объект"].astype("category").cat.codes
panel["тип_датчика_code"] = panel["тип_датчика"].astype("category").cat.codes

train, val, test = fe.split_by_time(panel)
drop_cols = (["ид_канала_данных", "бакет", "дата", "год_неделя", "аномально",
              "ид_объект", "тип_датчика", "серия_старт", "дни_до_серии", "дни_класс",
              "цель_6ч", "цель_12ч", "цель_24ч", "цель_48ч"]
             + [f"цель_серии_{n}" for n in fe.HORIZONS])
X_cols = [c for c in train.columns if c not in drop_cols]

print("Train:", train.shape, "| Val:", val.shape, "| Test:", test.shape)
print("Признаков:", len(X_cols))
for n in fe.HORIZONS:
    print(f"  цель_серии_{n}: base train={100 * train[f'цель_серии_{n}'].mean():.2f}% | "
          f"test={100 * test[f'цель_серии_{n}'].mean():.2f}%")

# 10.3 · Слой 1: бинарный прогноз «новое событие в ближайшие 6/12/24/48ч»

CatBoost (CPU), цели — `цель_серии_*` (только старты серий). Метрики на тесте
2026: PR-AUC, lift (PR-AUC / база), Precision@Recall≥0.5, F1@0.5, порог под
Precision≥0.7.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import (average_precision_score, precision_recall_curve,
                             f1_score, precision_score as ps, recall_score as rs)

ITERS, LR, DEPTH, ES = 500, 0.05, 6, 200
rows = []
preds_series = {}
for name in fe.HORIZONS:
    y_col = f"цель_серии_{name}"
    tr = train.dropna(subset=[y_col])
    va = val.dropna(subset=[y_col])
    te = test.dropna(subset=[y_col])
    m = CatBoostClassifier(iterations=ITERS, learning_rate=LR, depth=DEPTH, random_seed=42,
                           task_type="CPU", eval_metric="F1", early_stopping_rounds=ES, verbose=0)
    m.fit(Pool(tr[X_cols], tr[y_col]), eval_set=Pool(va[X_cols], va[y_col]))
    yp = m.predict_proba(te[X_cols])[:, 1]
    yt = te[y_col].astype(int).values
    preds_series[name] = yp
    base = yt.mean()
    ap = average_precision_score(yt, yp)
    pr, rc, _ = precision_recall_curve(yt, yp)
    p_r05 = float(pr[rc >= 0.5].max()) if (rc >= 0.5).any() else float("nan")
    f1 = float(f1_score(yt, (yp >= 0.5).astype(int)))
    best = None
    for thr in np.round(np.arange(0.10, 0.99, 0.01), 3):
        ypr = (yp >= thr).astype(int)
        p_, r_ = ps(yt, ypr, zero_division=0), rs(yt, ypr)
        if p_ >= 0.7 and (best is None or r_ > best[2]):
            best = (thr, p_, r_)
    rows.append({"горизонт": name, "база,%": round(100 * base, 2),
                 "PR-AUC": round(ap, 4), "lift": round(ap / base, 2),
                 "Prec@R>=.5": round(p_r05, 4), "F1@.5": round(f1, 4),
                 "thr@P>=.7": (None if best is None else best[0]),
                 "Rec@thr": (None if best is None else round(best[2], 3))})
    print(f"  {name}: PR-AUC={ap:.4f} lift={ap / base:.1f}x")
res = pd.DataFrame(rows)
print("\nСлой 1 (старты серий) — тест 2026:")
print(res.to_string(index=False))

# 10.4 · Слой 2: прогноз «через сколько дней произойдёт событие»

Ор-динальный мультикласс по классам `дни_класс` (≤0.5д, 0.5–1д, 1–2д, 2–3д,
3–7д, 7–14д, 14–30д, >30д/нет события). Ожидаемые дни = Σ p(класс)·mid(класс).
Оценка на тесте 2026 по строкам с реальным (нецензурным) событием: MAE,
медиана |ошибки|, Спирмен, точность класса ±1. Плюс связь слоёв: бинарная
вероятность события в 24ч vs ожидаемые дни.

In [ ]:
from catboost import CatBoostClassifier as CatBoostMC
from scipy.stats import spearmanr

X_tr = train[X_cols]
m_days = CatBoostMC(iterations=700, learning_rate=0.05, depth=6, random_seed=42,
                    loss_function="MultiClass", eval_metric="Accuracy",
                    early_stopping_rounds=200, verbose=0)
m_days.fit(Pool(X_tr, train["дни_класс"]),
           eval_set=Pool(val[X_cols], val["дни_класс"]))
proba = m_days.predict_proba(test[X_cols])
pred_cls = proba.argmax(axis=1)
exp_days = proba @ MID

y_true_cls = test["дни_класс"].values
obs = test["дни_до_серии"].notna().values  # нецензурные строки: есть реальное событие
true_d = test["дни_до_серии"].values[obs]
exp_o = exp_days[obs]
err = np.abs(exp_o - true_d)
mae = err.mean()
med = np.median(err)
corr = spearmanr(exp_o, true_d).statistic
acc1 = np.mean(np.abs(pred_cls[obs] - y_true_cls[obs]) <= 1)

print("Слой 2 (дни до события), тест 2026, нецензурные строки (n=%d):" % obs.sum())
print(f"  MAE (дней):            {mae:.2f}")
print(f"  медиана |ошибки|:      {med:.2f}")
print(f"  Спирмен(ожид., факт.): {corr:.4f}")
print(f"  доля класса +-1 бин:   {100 * acc1:.2f}%")
print(f"  цензурировано на тесте: {100 * (1 - obs.mean()):.2f}%")
print("\nСвязь слоёв: корреляция P(событие в 24ч) vs ожидаемые дни = %.3f"
      % np.corrcoef(preds_series["24ч"], exp_days)[0, 1])
print("\nОжидаемые дни по группам P(24ч):")
qm = pd.qcut(pd.Series(preds_series["24ч"]), 4, duplicates="drop")
print(pd.Series(exp_days).groupby(qm).mean().round(1).to_string())

# 10.5 · Выводы

- **Детекция события по первому эпизоду серии** снижает шум: база “событие в 24ч”
  11.4% вместо 37% для «любой неисправности». Последующие неисправности серии —
  продолжения, в таргете не учитываются (`серия_старт`, `цель_серии_*`).
- **Слой 1** — вероятность нового события в 6/12/24/48ч (порог под Precision≥0.7).
- **Слой 2** — ожидаемое число дней до следующего события (ор-динальный класс,
  `дни_класс`); для диспетчера: «вероятность + срок».
- Интерпретация: высокий P(24ч) → малые ожидаемые дни (связь слоёв отрицательная).